# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

C:\dev\llms\llm_engineering\week8\agents\deals.py:27: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(description, 'html.parser').get_text()
100%|██████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [02:05<00:00, 25.15s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: Corelle Sets Sale: 30% off + free shipping w/ $79\nDetails: Use promo code "SETS30" to get 30% off 12- and 16-piece dinnerware, cookware, and knife sets. Shipping adds $10.99 or orders of $79 or more ships for free. Shop Now at Corelle\nFeatures: \nURL: https://www.dealnews.com/Corelle-Sets-Sale-30-off-free-shipping-w-79/21768641.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Open-Box Bluetti AC50B 700W 448Wh Portable Power Station for $243 w/ clip coupon + free shipping
Details: Clip the $24 off coupon on the landing page to drop the price to $242.82 in-cart. That's the best deal we've seen in any condition. Plus, it ships free from a US warehouse. Buy Now at AliExpress
Features: 3,500+ cycles 10-year lifespan Model: AC50B
URL: https://

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Bluetti AC180P-US-GY-BL-THDUS is a high-capacity portable power station offering 1,800W output and an impressive 1,024Wh battery capacity. This unit is equipped with a built-in MPPT charge controller for efficient solar charging, making it ideal for outdoor adventures or as affordable backup power. Its multiple output options support a variety of devices, ensuring you stay powered wherever you go.', price=433.0, url='https://www.dealnews.com/products/Bluetti/Bluetti-1-800-W-Li-Fe-PO4-Solar-Portable-Power-Station/482799.html?iref=rss-c142')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description="The Bluetti AC50B Portable Power Station is a versatile device featuring a powerful 700W output and a 448Wh capacity, making it perfect for outdoor adventures and emergency situations. With over 3,500 charge cycles and a 10-year lifespan, it's designed for long-term use. Equipped with multiple ports, it allows you to charge a variety of devices simultaneously. The power station is compact and portable, making it easy to transport wherever you need energy on-the-go.", price=243.0, url='https://www.dealnews.com/products/Bluetti/Bluetti-AC50-B-700-W-448-Wh-Portable-Power-Station/481242.html?iref=rss-c142'), Deal(product_description="The Bluetti AC180P is a robust 1,800W LiFePO4 Solar Portable Power Station, equipped with a built-in MPPT charge controller for efficient solar energy usage. With a high capacity designed for multiple devices, it's ideal for both emergency power backup and outdoor activities. This model offers over 3,500 cycles an